In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS workspace.gold COMMENT 'Capa gold'")

In [0]:
spark.sql("DROP TABLE IF EXISTS workspace.gold.dim_tiendas")

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.gold.dim_tiendas (
  id_tienda LONG,
  nombre_tienda STRING
)
""")


In [0]:
import pandas as pd

df = spark.table("workspace.silver.adidas_us_sales").toPandas()

# Se crea la dimension de tiendas, desde la tabla de ventas
dim_tiendas = df[['tienda']].drop_duplicates().reset_index(drop=True)
dim_tiendas['id_tienda'] = dim_tiendas.index + 1
dim_tiendas = dim_tiendas.rename(columns={'tienda': 'nombre_tienda'})
dim_tiendas = dim_tiendas[['id_tienda', 'nombre_tienda']]

df_spark = spark.createDataFrame(dim_tiendas)


In [0]:

# Usamos overwrite para reemplazar completamente los datos existentes
df_spark.write.format("delta") \
    .option("mergeSchema", "true") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.dim_tiendas")